# Convolutional Neural Network for Dyson EPR Line Parameters Estimation

**Authors**: A.V. Uriadov, D.V. Savchenko

*National Technical University of Ukraine "Igor Sikorsky Kyiv Polytechnic Institute"*


# 0. Execution Environment Resource Check for EPR-CNN

Performs an extended diagnostics of the Google Colab runtime to assess its suitability for training an **EPR-CNN model**.

- **Checks GPU availability and type**  
  Detects NVIDIA GPU, identifies its model (T4, V100, A100, etc.), memory usage, driver and CUDA version, and provides performance-based recommendations (e.g. batch size).
- **Verifies ML frameworks**  
  Inspects installed versions of **TensorFlow** and **Keras**, and checks whether GPU acceleration is enabled.
- **Analyzes RAM usage**  
  Reports total, available, and used RAM, flags high memory pressure, and issues practical recommendations.
- **Monitors memory leaks**  
  Sets up a lightweight RAM monitor to detect abnormal memory growth during or after training.
- **Suggests optimal EPR-CNN settings**  
  Automatically recommends batch size, model complexity, mixed precision, and sequence length based on available GPU and RAM.
- **Provides a final system score**  
  Outputs a simple quantitative rating of how well the system is suited for EPR-CNN training.

## Purpose
This section helps ensure **reproducible, efficient, and stable training** of EPR-CNN models by adapting training parameters to the actual hardware resources of the runtime.

In [ ]:
import sys, tensorflow as tf, numpy as np, scipy, matplotlib

print("Python", sys.version.split()[0])
print("TensorFlow", tf.__version__)
print("NumPy", np.__version__)
print("SciPy", scipy.__version__)
print("Matplotlib", matplotlib.__version__)

In [ ]:
# =============================================================================
# EXECUTION ENVIRONMENT RESOURCE CHECK FOR MODEL CREATION AND TRAINING
# =============================================================================

# Install required libraries
!pip install psutil gpustat

# Import modules
import subprocess
import sys
import platform
from psutil import virtual_memory
import warnings
import re
import json
import time
import gc

def get_gpu_detailed_info():
    """
    Extended GPU check with detailed information about type and characteristics.
    Returns:
        dict: Detailed GPU information
    """
    gpu_info = {
        'available': False,
        'name': 'Unknown',
        'type': 'Unknown',  # T4, V100, A100, K80, P100
        'memory_total': 0,
        'memory_free': 0,
        'memory_used': 0,
        'driver_version': 'Unknown',
        'cuda_version': 'Unknown',
        'temperature': 0,
        'utilization': 0,
        'compute_capability': 'Unknown',
        'performance_rating': 'Unknown',
        'recommended_batch_size': 0,
        'error_message': None
    }

    # GPU characteristics dictionary for EPR-CNN
    gpu_specs = {
        'T4': {'performance': 'Good', 'batch_size': 32, 'compute': '7.5'},
        'V100': {'performance': 'Excellent', 'batch_size': 64, 'compute': '7.0'},
        'A100': {'performance': 'Best', 'batch_size': 128, 'compute': '8.0'},
        'K80': {'performance': 'Basic', 'batch_size': 16, 'compute': '3.7'},
        'P100': {'performance': 'Very good', 'batch_size': 48, 'compute': '6.0'},
        'P4': {'performance': 'Basic', 'batch_size': 24, 'compute': '6.1'},
        'RTX': {'performance': 'Excellent', 'batch_size': 64, 'compute': '7.5+'}
    }

    try:
        # Try to retrieve detailed information via nvidia-smi query
        try:
            result = subprocess.run(
                ['nvidia-smi',
                 '--query-gpu=name,memory.total,memory.free,memory.used,temperature.gpu,utilization.gpu,driver_version',
                 '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=10
            )
            if result.returncode == 0:
                lines = result.stdout.strip().split('\n')
                if lines and lines[0]:
                    data = lines[0].split(', ')
                    if len(data) >= 6:
                        gpu_info['available'] = True
                        gpu_info['name'] = data[0].strip()
                        gpu_info['memory_total'] = int(data[1])
                        gpu_info['memory_free'] = int(data[2])
                        gpu_info['memory_used'] = int(data[3])
                        gpu_info['temperature'] = int(data[4]) if data[4].isdigit() else 0
                        gpu_info['utilization'] = int(data[5]) if data[5].isdigit() else 0
                        gpu_info['driver_version'] = data[6] if len(data) > 6 else 'Unknown'
        except:
            pass

        # If that failed, fall back to standard nvidia-smi
        if not gpu_info['available']:
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=15)
            if result.returncode == 0:
                output = result.stdout
                if 'nvidia' in output.lower() and not any(err in output.lower() for err in ['failed', 'error', 'not found']):
                    gpu_info['available'] = True

                    # Parse GPU name
                    for line in output.split('\n'):
                        if any(gpu_type in line.upper() for gpu_type in ['T4', 'V100', 'A100', 'K80', 'P100', 'P4', 'RTX']):
                            gpu_info['name'] = line.strip()
                            break

        # Determine GPU type and characteristics
        if gpu_info['available']:
            gpu_name_upper = gpu_info['name'].upper()
            for gpu_type, specs in gpu_specs.items():
                if gpu_type in gpu_name_upper:
                    gpu_info['type'] = gpu_type
                    gpu_info['performance_rating'] = specs['performance']
                    gpu_info['recommended_batch_size'] = specs['batch_size']
                    gpu_info['compute_capability'] = specs['compute']
                    break

        # Retrieve CUDA version
        try:
            cuda_result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, timeout=5)
            if cuda_result.returncode == 0:
                cuda_match = re.search(r'release (\d+\.\d+)', cuda_result.stdout)
                if cuda_match:
                    gpu_info['cuda_version'] = cuda_match.group(1)
        except:
            pass

    except Exception as e:
        gpu_info['error_message'] = f'Error: {str(e)}'

    return gpu_info

def check_ml_frameworks():
    """
    Checks TensorFlow versions.
    Returns:
        dict: Information about ML frameworks
    """
    frameworks_info = {
        'tensorflow': {'installed': False, 'version': 'Not installed', 'gpu_support': False},
        'keras': {'installed': False, 'version': 'Not installed'},
        'recommendations': []
    }

    # TensorFlow check
    try:
        import tensorflow as tf
        frameworks_info['tensorflow']['installed'] = True
        frameworks_info['tensorflow']['version'] = tf.__version__

        # Check GPU support in TensorFlow
        try:
            frameworks_info['tensorflow']['gpu_support'] = len(tf.config.list_physical_devices('GPU')) > 0
        except:
            frameworks_info['tensorflow']['gpu_support'] = False

        # Version recommendations for TensorFlow
        tf_version = tf.__version__
        if tf_version.startswith('2.'):
            frameworks_info['recommendations'].append("[INFO] TensorFlow 2.x detected (modern version).")
        else:
            frameworks_info['recommendations'].append("[WARN] Older TensorFlow version detected; upgrade is recommended.")

    except ImportError:
        frameworks_info['recommendations'].append("[ERROR] TensorFlow is not installed.")

    try:
        import keras
        frameworks_info['keras']['installed'] = True
        frameworks_info['keras']['version'] = keras.__version__
    except ImportError:
        pass

    return frameworks_info

def check_ram_availability():
    """
    Checks RAM availability and returns detailed information.
    Returns:
        dict: Dictionary with RAM information
    """
    ram_info = {
        'total_gb': 0,
        'available_gb': 0,
        'used_gb': 0,
        'usage_percent': 0,
        'is_high_ram': False,
        'recommendation': ''
    }

    try:
        memory = virtual_memory()
        ram_info['total_gb'] = memory.total / 1e9
        ram_info['available_gb'] = memory.available / 1e9
        ram_info['used_gb'] = memory.used / 1e9
        ram_info['usage_percent'] = memory.percent
        ram_info['is_high_ram'] = ram_info['total_gb'] >= 20

        # Recommendations
        if ram_info['usage_percent'] > 80:
            ram_info['recommendation'] = '[WARN] High RAM usage detected; runtime restart is recommended.'
        elif ram_info['total_gb'] < 8:
            ram_info['recommendation'] = '[WARN] Low RAM detected; performance may be limited.'
        else:
            ram_info['recommendation'] = '[INFO] RAM status is within normal limits.'

    except Exception as e:
        ram_info['recommendation'] = f'[ERROR] Failed to retrieve RAM information: {str(e)}'

    return ram_info

def setup_memory_monitoring():
    """
    Sets up memory monitoring to warn about memory leaks.
    """
    import gc

    # Baseline memory state
    initial_ram = virtual_memory().used / 1e9

    def check_memory_leak(threshold_gb=2.0):
        """
        Checks for memory leaks.
        Args:
            threshold_gb: Memory increase threshold in GB
        """
        current_ram = virtual_memory().used / 1e9
        increase = current_ram - initial_ram

        if increase > threshold_gb:
            print("[WARN] Possible memory leak detected.")
            print(f"[WARN] RAM increase: +{increase:.1f} GB")
            print("[INFO] Recommended actions:")
            print("[INFO] - Run gc.collect()")
            print("[INFO] - Use del for large variables")
            print("[INFO] - Consider reducing batch_size")

            # Automatic cleanup
            collected = gc.collect()
            print(f"[INFO] Garbage collector removed objects: {collected}")

        return increase

    return check_memory_leak

def get_optimal_settings_for_epr_cnn(gpu_info, ram_gb):
    """
    Recommends optimal settings for EPR-CNN based on available resources.
    """
    settings = {
        'batch_size': 16,
        'model_complexity': 'simple',
        'data_parallel': False,
        'mixed_precision': False,
        'gradient_accumulation': 1,
        'max_sequence_length': 1024,
        'warnings': []
    }

    # GPU-based settings
    if gpu_info['available']:
        gpu_type = gpu_info['type']
        memory_gb = gpu_info['memory_total'] / 1024 if gpu_info['memory_total'] > 0 else 0

        if gpu_type in ['A100', 'V100']:
            settings['batch_size'] = 64
            settings['model_complexity'] = 'complex'
            settings['mixed_precision'] = True
            if memory_gb > 20:
                settings['max_sequence_length'] = 2048
        elif gpu_type == 'T4':
            settings['batch_size'] = 32
            settings['model_complexity'] = 'medium'
            settings['mixed_precision'] = True
        elif gpu_type in ['K80', 'P4']:
            settings['batch_size'] = 16
            settings['model_complexity'] = 'simple'
            settings['warnings'].append("[WARN] Legacy GPU detected; training may be slow.")

    # RAM-based adjustments
    if ram_gb < 8:
        settings['batch_size'] = min(settings['batch_size'], 16)
        settings['warnings'].append("[WARN] Low RAM detected; batch_size reduced.")
    elif ram_gb > 25:
        settings['gradient_accumulation'] = 2

    return settings

# =============================================================================
# RUNNING EXTENDED CHECKS
# =============================================================================

# Detailed GPU check
print("[INFO] GPU diagnostics:")
gpu_status = get_gpu_detailed_info()

if gpu_status['available']:
    print("[INFO] GPU detected and available.")
    print(f"[INFO] GPU name: {gpu_status['name']}")
    print(f"[INFO] GPU type: {gpu_status['type']}")
    print(f"[INFO] Performance rating: {gpu_status['performance_rating']}")
    print(f"[INFO] Recommended batch_size: {gpu_status['recommended_batch_size']}")
    print(f"[INFO] Compute capability: {gpu_status['compute_capability']}")
    if gpu_status['memory_total'] > 0:
        memory_usage = (gpu_status['memory_used'] / gpu_status['memory_total']) * 100
        print(f"[INFO] GPU memory usage: {gpu_status['memory_used']} MB / {gpu_status['memory_total']} MB ({memory_usage:.1f}%)")
    print(f"[INFO] NVIDIA driver version: {gpu_status['driver_version']}")
    print(f"[INFO] CUDA version: {gpu_status['cuda_version']}")
else:
    print("[ERROR] No GPU detected or GPU is not available.")

# ML frameworks check
print("[INFO] ML framework diagnostics:")
frameworks = check_ml_frameworks()

print(f"[INFO] TensorFlow: {'installed' if frameworks['tensorflow']['installed'] else 'not installed'} ({frameworks['tensorflow']['version']})")
if frameworks['tensorflow']['installed']:
    print(f"[INFO] TensorFlow GPU support: {'enabled' if frameworks['tensorflow']['gpu_support'] else 'disabled'}")

print(f"[INFO] Keras: {'installed' if frameworks['keras']['installed'] else 'not installed'} ({frameworks['keras']['version']})")

for rec in frameworks['recommendations']:
    print(rec)

# RAM check
print("[INFO] RAM diagnostics:")
ram_status = check_ram_availability()
print(f"[INFO] Total RAM: {ram_status['total_gb']:.1f} GB")
print(f"[INFO] Available RAM: {ram_status['available_gb']:.1f} GB")
print(f"[INFO] RAM usage: {ram_status['usage_percent']:.1f}%")
print(ram_status['recommendation'])

# Memory monitoring setup
print("[INFO] Configuring memory leak monitoring:")
memory_checker = setup_memory_monitoring()
print("[INFO] Memory leak monitoring is enabled.")
print("[INFO] Call memory_checker() after training to evaluate memory growth.")

# Optimal settings for EPR-CNN
print("[INFO] Recommended training settings for EPR-CNN:")
optimal_settings = get_optimal_settings_for_epr_cnn(gpu_status, ram_status['total_gb'])

print(f"[INFO] batch_size: {optimal_settings['batch_size']}")
print(f"[INFO] model_complexity: {optimal_settings['model_complexity']}")
print(f"[INFO] mixed_precision: {'enabled' if optimal_settings['mixed_precision'] else 'disabled'}")
print(f"[INFO] gradient_accumulation: {optimal_settings['gradient_accumulation']}")
print(f"[INFO] max_sequence_length: {optimal_settings['max_sequence_length']}")

for warning in optimal_settings['warnings']:
    print(warning)

# Final system evaluation
print("[INFO] Overall system evaluation for EPR-CNN:")
score = 0
if gpu_status['available']:
    score += 40

if frameworks['tensorflow']['gpu_support']:
    score += 30

if ram_status['total_gb'] >= 12:
    score += 20
if gpu_status['type'] in ['T4', 'V100', 'A100']:
    score += 10

if score >= 90:
    print("[INFO] Result: EXCELLENT. System is ideal for EPR-CNN training.")
elif score >= 70:
    print("[INFO] Result: GOOD. System is suitable for EPR-CNN training.")
elif score >= 50:
    print("[WARN] Result: SATISFACTORY. Performance limitations are possible.")
else:
    print("[ERROR] Result: INSUFFICIENT. Configuration upgrade is recommended.")

print("------------------------------------------------")
print(f"[INFO] Overall score: {score}/100")
print("------------------------------------------------")


# Export settings for further use
print("[INFO] Settings exported to variables:")
print("[INFO] - gpu_status: GPU information")
print("[INFO] - optimal_settings: recommended parameters")
print("[INFO] - memory_checker: memory monitoring function")


# 1. Loading of traning data



## 1.1. Safe Google Drive Mounting

This subsection provides a **reliable and idempotent Google Drive mounting routine** for Google Colab.

The function checks whether `MyDrive` is already accessible, avoids conflicts with non-empty mount points, and supports clean remounting when required.  
Successful mounting is validated explicitly to ensure stable access to persistent storage.

Intended for **production notebooks** and long-running experiments where robust filesystem access is required.

In [ ]:
import os
from google.colab import drive

def safe_mount_gdrive(mountpoint="/content/drive", force=False):
    """
    Safely mount Google Drive in Colab.
    - Checks whether 'MyDrive' is accessible in the mount point
    - Mounts Drive if it is not already available
    """
    mydrive_path = os.path.join(mountpoint, "MyDrive")

    if os.path.isdir(mydrive_path) and len(os.listdir(mydrive_path)) > 0:
        print(f"[INFO] Google Drive is already mounted at: {mountpoint}")
        return mountpoint

    # If there are existing files — force remount
    if os.path.exists(mountpoint) and os.listdir(mountpoint):
        if force:
            print("[WARN] Mount point is not empty; forcing remount.")
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            os.system(f"rm -rf {mountpoint}")
        else:
            print("[INFO] Mount point is not empty; using alternative mount point: /content/gdrive")
            mountpoint = "/content/gdrive"
            os.makedirs(mountpoint, exist_ok=True)

    # Mounting
    drive.mount(mountpoint, force_remount=force)

    # Success check
    mydrive_path = os.path.join(mountpoint, "MyDrive")
    if os.path.isdir(mydrive_path):
        print(f"[INFO] Google Drive mounted successfully at: {mountpoint}")
    else:
        print("[ERROR] Failed to detect the MyDrive folder; check permissions.")

    return mountpoint

# --- Usage ---
mp = safe_mount_gdrive("/content/drive")
# --- Check ---
# !ls -la {mp}/MyDrive | head -n 20

## 1.2. Loading Synthetic MIX Dataset

This subsection loads a **synthetic mixed Dysonian dataset** generated by `DysonGeneratorMix`.

It validates the presence of all required files, loads input spectra, target parameters, and the magnetic field axis, and performs basic consistency and sanity checks.  
Minimum and maximum values of the target parameters are computed locally for later normalization.  
Metadata is also loaded to provide context for the dataset configuration and parameter interpretation.

The cell is intended for **production use**, with explicit checks to ensure data integrity before model training.

In [ ]:
# ===========================
# Load synthetic dataset (single "mix" dataset)
# ===========================
import os, json
import numpy as np

work_dir = f"{mp}/MyDrive/Python/DysonianLineCNN-mix"

PREFIX = "dataset"  # matches the Prefix parameter in DysonGeneratorMix.m

paths = {
    "X":      os.path.join(work_dir, f"X_dyson_mix_{PREFIX}.npy"),
    "y":      os.path.join(work_dir, f"y_dyson_mix_{PREFIX}.npy"),
    "B_axis": os.path.join(work_dir, f"B_axis_mix_{PREFIX}.npy"),
    "meta":   os.path.join(work_dir, f"meta_mix_{PREFIX}.json"),
}

# file existence check
for key, p in paths.items():
    assert os.path.exists(p), f"[ERROR] Missing file for {key}: {p}"

# load arrays
X      = np.load(paths["X"]).astype(np.float32)                 # (N, Npoints)
y      = np.load(paths["y"]).astype(np.float32)                 # (N, 3) -> [B0, dB, p3]
B_axis = np.load(paths["B_axis"]).astype(np.float32).squeeze()  # (Npoints,)

# y_min / y_max (computed locally)
y_min = np.min(y, axis=0, keepdims=True).astype(np.float32)  # (1, 3)
y_max = np.max(y, axis=0, keepdims=True).astype(np.float32)  # (1, 3)

# meta (useful for debugging: mode, Npoints, p3 meaning)
with open(paths["meta"], "r", encoding="utf-8") as f:
    meta = json.load(f)

# sanity checks
assert X.ndim == 2, f"[ERROR] X must be 2D (N, Npoints), got {X.shape}"
assert y.ndim == 2 and y.shape[1] == 3, f"[ERROR] y shape must be (N, 3), got {y.shape}"
assert X.shape[0] == y.shape[0], "[ERROR] X and y must have the same number of samples"
assert B_axis.ndim == 1 and B_axis.shape[0] == X.shape[1], f"[ERROR] B_axis must match Npoints={X.shape[1]}, got {B_axis.shape}"
if np.isnan(X).any() or np.isinf(X).any(): raise ValueError("[ERROR] X contains NaN/Inf values")
if np.isnan(y).any() or np.isinf(y).any(): raise ValueError("[ERROR] y contains NaN/Inf values")

print("[INFO] MIX dataset loaded successfully.")
print("[INFO] X.shape =", X.shape)
print("[INFO] y.shape =", y.shape, "(B0, dB, p3)")
print(f"[INFO] B_axis  = {B_axis.shape} [{float(B_axis[0]):.1f} .. {float(B_axis[-1]):.1f}] G")
print("[INFO] y_min   =", y_min.flatten())
print("[INFO] y_max   =", y_max.flatten())
print("[INFO] meta.mode =", meta.get("mode"), "| p3 meaning:", meta.get("third_param_meaning"))

# 2. Dataset Splitting and Normalization (3-channel input: +B_axis)


In [ ]:
# (3-channel input: normalized signal + B_axis)

import time
import os
import numpy as np
from sklearn.model_selection import train_test_split

Npoints = X.shape[1]

# 1) Train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, shuffle=True
)

# 2) Per-sample normalization (two signal channels)
def standardize_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    mu = np.mean(Xarr, axis=1, keepdims=True)
    sd = np.std(Xarr, axis=1, keepdims=True) + eps
    return (Xarr - mu) / sd

def ptp_norm_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    xmin = Xarr.min(axis=1, keepdims=True)
    ptp  = (Xarr.max(axis=1, keepdims=True) - xmin) + eps
    return (Xarr - xmin) / ptp * 2.0 - 1.0   # [-1, 1]

def make_two_channel(Xarr):
    ch0 = standardize_per_sample(Xarr)         # (N, Npoints)
    ch1 = ptp_norm_per_sample(Xarr)            # (N, Npoints)
    return np.stack([ch0, ch1], axis=-1)       # (N, Npoints, 2)

# === 2.5) Add 3rd channel: B_axis (physical magnetic field axis) ===
# IMPORTANT: B_axis must have shape (Npoints,) or (1, Npoints).
# If not present, load or construct B_axis BEFORE this step.
assert 'B_axis' in globals(), "[ERROR] B_axis is not defined. Load B_axis before Step 2."
assert len(B_axis) == Npoints, f"[ERROR] B_axis length mismatch: expected {Npoints}, got {len(B_axis)}"

def make_three_channel_Baxis(Xarr, B_axis):
    two = make_two_channel(Xarr)  # (N, Npoints, 2)

    b = np.asarray(B_axis, dtype=np.float32).reshape(1, -1)  # (1, Npoints)
    b = (b - b.min()) / (b.max() - b.min() + 1e-8)           # [0, 1]
    b = b * 2.0 - 1.0                                        # [-1, 1]
    ch2 = np.repeat(b, Xarr.shape[0], axis=0)                # (N, Npoints)

    return np.concatenate([two, ch2[..., None]], axis=-1).astype(np.float32)  # (N, Npoints, 3)

# === 2.6) Build network inputs (3 channels total) ===
X_train_in = make_three_channel_Baxis(X_train, B_axis)
X_val_in   = make_three_channel_Baxis(X_val,   B_axis)
X_test_in  = make_three_channel_Baxis(X_test,  B_axis)

# === 3) Target normalization to [0, 1] using TRAIN set only (avoid data leakage) ===
y_min_local = np.min(y_train, axis=0, keepdims=True).astype(np.float32)
y_max_local = np.max(y_train, axis=0, keepdims=True).astype(np.float32)

def minmax_apply(yarr, ymin, ymax):
    return (yarr.astype(np.float32) - ymin) / (ymax - ymin + 1e-8)

y_train_norm = minmax_apply(y_train, y_min_local, y_max_local)
y_val_norm   = minmax_apply(y_val,   y_min_local, y_max_local)
y_test_norm  = minmax_apply(y_test,  y_min_local, y_max_local)

def out_of_range_stats(y_norm):
    below = (y_norm < 0).mean()
    above = (y_norm > 1).mean()
    return below, above

b0, a0 = out_of_range_stats(y_val_norm)
b1, a1 = out_of_range_stats(y_test_norm)

print("[INFO] y_min (train):", y_min_local.flatten())
print("[INFO] y_max (train):", y_max_local.flatten())
print(f"[INFO] Validation out-of-range: below0={b0:.4f}, above1={a0:.4f}")
print(f"[INFO] Test out-of-range:       below0={b1:.4f}, above1={a1:.4f}")

# Clip to [0, 1] for numerical stability
y_train_norm = np.clip(y_train_norm, 0.0, 1.0)
y_val_norm   = np.clip(y_val_norm,   0.0, 1.0)
y_test_norm  = np.clip(y_test_norm,  0.0, 1.0)

# 4) Shape checks
print(f"[INFO] Train set:       X {X_train_in.shape}, y {y_train_norm.shape}")
print(f"[INFO] Validation set:  X {X_val_in.shape},   y {y_val_norm.shape}")
print(f"[INFO] Test set:        X {X_test_in.shape},  y {y_test_norm.shape}")

# 5) Target dictionaries for multi-output model
y_train_dict = {
    'B0': y_train_norm[:, 0],
    'dB': y_train_norm[:, 1],
    'p3': y_train_norm[:, 2]
}
y_val_dict = {
    'B0': y_val_norm[:, 0],
    'dB': y_val_norm[:, 1],
    'p3': y_val_norm[:, 2]
}

# Note for model definition:
# input_shape = (Npoints, 3)
stamp = time.strftime('%Y%m%d_%H%M%S')
run_dir = os.path.join(work_dir, "runs", stamp)
os.makedirs(run_dir, exist_ok=True)

np.savetxt(
    os.path.join(run_dir, "B_axis.csv"),
    B_axis.reshape(-1),
    delimiter=","
)

print("[INFO] B_axis.csv saved successfully.")

# 3. Dysonian CNN Model Definition

This section defines a **multi-output 1D convolutional neural network** for regression of Dysonian EPR spectra parameters.  
The model uses residual and dilated convolutional blocks to capture multi-scale spectral features and predicts the parameters **B₀**, **ΔB**, and **p₃** simultaneously.

Mixed-precision training is enabled when available to improve performance, and the network is compiled with weighted losses to emphasize the physically sensitive **p₃** parameter.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers as L, models, optimizers, losses, metrics

# Enable mixed precision if available
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("[INFO] Mixed precision policy set to mixed_float16.")
except Exception as e:
    print("[WARN] Mixed precision could not be enabled:", e)

# Set global random seed for reproducibility
tf.keras.utils.set_random_seed(42)

def _res_block(x, filters, k=3, dilation=1):
    """
    Residual 1D convolutional block with optional dilation.
    """
    y = L.Conv1D(filters, k, padding="same", dilation_rate=dilation, activation="relu")(x)
    y = L.Conv1D(filters, k, padding="same", dilation_rate=dilation, activation=None)(y)
    if x.shape[-1] != filters:
        x = L.Conv1D(filters, 1, padding="same", activation=None)(x)
    y = L.Add()([x, y])
    y = L.Activation("relu")(y)
    return y

def build_dysonian_model(input_shape, dropout=0.15, p3_weight=2.0):
    """
    Build a multi-output 1D CNN for Dysonian EPR spectra regression.
    Outputs: B0, dB, p3
    """
    inp = L.Input(shape=input_shape, name="EPR_input")

    x = L.Conv1D(32, 9, padding="same", activation="relu")(inp)
    x = L.Conv1D(48, 7, padding="same", activation="relu")(x)

    x = L.MaxPooling1D(2)(x)
    x = _res_block(x, 64,  k=5, dilation=1)
    x = _res_block(x, 64,  k=5, dilation=2)

    x = L.MaxPooling1D(2)(x)
    x = _res_block(x, 96,  k=5, dilation=2)
    x = _res_block(x, 96,  k=3, dilation=4)

    x = L.MaxPooling1D(2)(x)
    x = _res_block(x, 128, k=3, dilation=4)
    x = _res_block(x, 128, k=3, dilation=8)

    x = L.GlobalAveragePooling1D()(x)
    x = L.Dense(512, activation="relu")(x)
    x = L.Dropout(dropout)(x)

    def head(name):
        h = L.Dense(128, activation="relu")(x)
        # Output cast to float32 for numerical stability under mixed precision
        return L.Dense(1, name=name, activation="linear", dtype="float32")(h)

    out_B0 = head("B0")
    out_dB = head("dB")
    out_p3 = head("p3")

    model = models.Model(
        inp,
        [out_B0, out_dB, out_p3],
        name="DysonCNN_mix"
    )

    opt = optimizers.Adam(learning_rate=1e-3)
    model.compile(
        optimizer=opt,
        loss={
            "B0": losses.MeanSquaredError(),
            "dB": losses.MeanSquaredError(),
            "p3": losses.MeanSquaredError()
        },
        loss_weights={
            "B0": 1.0,
            "dB": 1.0,
            "p3": float(p3_weight)
        },
        metrics={
            "B0": metrics.MeanAbsoluteError(name="mae"),
            "dB": metrics.MeanAbsoluteError(name="mae"),
            "p3": metrics.MeanAbsoluteError(name="mae")
        }
    )
    return model

# Npoints is defined in the previous cell: Npoints = X.shape[1]
model = build_dysonian_model(
    input_shape=(Npoints, 3),
    dropout=0.15,
    p3_weight=2.0
)

print("[INFO] Dysonian CNN model built successfully.")
model.summary()

# 4. Model Training and Artifact Saving

This section trains the Dysonian CNN using the prepared datasets with early stopping and adaptive learning-rate scheduling.  
Training and validation losses are monitored, and learning-rate changes are logged for traceability.

After training, the model, normalization parameters, metadata, training history, and diagnostic plots are saved to a timestamped run directory, ensuring full reproducibility of the experiment.

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

assert 'model' in globals(), "[ERROR] Model is not defined. Build the model first (Step 3)."
for name in ['X_train_in','X_val_in','X_test_in','y_train_norm','y_val_norm','y_test_norm']:
    assert name in globals(), f"[ERROR] Missing variable in memory: {name}"

# ensure output head keys are consistent
y_train_dict = {"B0": y_train_norm[:,0], "dB": y_train_norm[:,1], "p3": y_train_norm[:,2]}
y_val_dict   = {"B0": y_val_norm[:,0],   "dB": y_val_norm[:,1],   "p3": y_val_norm[:,2]}

# compile (ensure p3 weighting is consistent)
opt = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
model.compile(
    optimizer=opt,
    loss={"B0":"mse","dB":"mse","p3":"mse"},
    loss_weights={"B0":1.0, "dB":1.0, "p3":2.0},
    metrics={"B0":"mae","dB":"mae","p3":"mae"}
)
print("[INFO] Initial learning rate:", float(tf.keras.backend.get_value(model.optimizer.learning_rate)))

# callbacks
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=40, min_delta=1e-6,
    restore_best_weights=True, verbose=1
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
)
class LrLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        print(f"[INFO] Epoch {epoch+1}: lr = {lr:.6e}")

# training
history = model.fit(
    X_train_in, y_train_dict,
    validation_data=(X_val_in, y_val_dict),
    epochs=300,
    batch_size=64,
    callbacks=[early_stop, reduce_lr, LrLogger()],
    verbose=1
)

# plot loss
plt.figure(figsize=(7.5,5))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epoch'); plt.ylabel('Loss (MSE)')
plt.title('Training / Validation Loss'); plt.grid(True); plt.legend(); plt.show()

# save artifacts
# stamp = time.strftime('%Y%m%d_%H%M%S')
run_dir = os.path.join(work_dir, "runs", stamp)
os.makedirs(run_dir, exist_ok=True)

model.save(os.path.join(run_dir, 'cnn_model.keras'))

np.save(os.path.join(run_dir, 'y_min.npy'), y_min_local.astype('float32'))
np.save(os.path.join(run_dir, 'y_max.npy'), y_max_local.astype('float32'))
np.save(os.path.join(run_dir, 'B_axis.npy'), B_axis.astype('float32'))

# If generator metadata is available in memory, include it in the output
generator_meta = globals().get("meta", None)

meta_out = {
    "created": stamp,
    "dataset": globals().get("PREFIX", "unknown_prefix"),
    "input_shape": list(X_train_in.shape[1:]),
    "heads": ["B0","dB","p3"],
    "loss_weights": {"B0":1.0,"dB":1.0,"p3":2.0},
    "y_min": y_min_local.flatten().tolist(),
    "y_max": y_max_local.flatten().tolist(),
    "generator_meta": generator_meta,
}
with open(os.path.join(run_dir, 'model_meta.json'), 'w', encoding="utf-8") as f:
    json.dump(meta_out, f, indent=2, ensure_ascii=False)

import pandas as pd
pd.DataFrame(history.history).to_csv(os.path.join(run_dir, 'history.csv'), index=False)

# IMPORTANT: call savefig before show, or save again explicitly after rendering
plt.figure(figsize=(7.5,5))
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch'); plt.ylabel('Loss (MSE)')
plt.title('Training / Validation Loss'); plt.grid(True); plt.legend()
plt.savefig(os.path.join(run_dir, "loss.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\n[INFO] Artifacts saved to:", run_dir)

## Training / Validation Loss (full and zoomed tail)

In [ ]:
loss     = history.history['loss']
val_loss = history.history['val_loss']

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# (a) Training / Validation (full)
ax[0].plot(loss, label='Training')
ax[0].plot(val_loss, label='Validation')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Training / Validation Loss (full)')
ax[0].grid(True)
ax[0].legend()

# (b) Zoomed tail
ax[1].plot(loss, label='Training')
ax[1].plot(val_loss, label='Validation')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Loss (MSE)')
ax[1].set_title('Training / Validation Loss (zoomed tail)')
ax[1].set_ylim(0.0, 0.01)
ax[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(run_dir, "loss_full_and_zoom.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# 5. Model Evaluation and Result Denormalization

This section evaluates the trained Dysonian CNN on the test dataset using a three-channel input representation that includes the physical magnetic field axis.  
The latest trained model is automatically loaded, predictions are generated in normalized space, and then denormalized back to physical units.

Performance is quantified using MAE and RMSE for each predicted parameter, and parity plots are generated to visualize agreement between true and predicted values.  
All evaluation artifacts, including plots and prediction tables, are saved for further analysis.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow import keras

# ---------------------------
# 0) Sanity checks: required objects
# ---------------------------
assert 'work_dir' in globals() and os.path.isdir(work_dir), "[ERROR] work_dir not found or is not a directory."
assert 'y_test_norm' in globals(), "[ERROR] y_test_norm is not available in memory."
assert y_test_norm.ndim == 2 and y_test_norm.shape[1] == 3, \
    f"[ERROR] Expected y_test_norm shape (N,3), got {y_test_norm.shape}"

# Required for the 3rd channel
assert 'B_axis' in globals(), "[ERROR] B_axis is not available in memory (required for the 3rd channel)."
B_axis = np.asarray(B_axis, dtype=np.float32).reshape(-1)

# ---------------------------
# 1) Build X_test_norm (guaranteed shape: (N, Npoints, 3))
# ---------------------------
def standardize_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    mu = Xarr.mean(axis=1, keepdims=True)
    sd = Xarr.std(axis=1, keepdims=True) + eps
    return (Xarr - mu) / sd

def ptp_norm_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    xmin = Xarr.min(axis=1, keepdims=True)
    ptp  = (Xarr.max(axis=1, keepdims=True) - xmin) + eps
    return (Xarr - xmin) / ptp * 2.0 - 1.0  # [-1, 1]

def baxis_channel(B_axis_1d, N, eps=1e-8):
    b = np.asarray(B_axis_1d, dtype=np.float32).reshape(1, -1)  # (1, Npoints)
    b = (b - b.min()) / (b.max() - b.min() + eps)               # [0, 1]
    b = b * 2.0 - 1.0                                           # [-1, 1]
    return np.repeat(b, N, axis=0)                               # (N, Npoints)

# Preferred source of raw test data: X_test (N, Npoints)
# Fallback: X_test_in only if it is already 3-channel
if 'X_test' in globals() and X_test is not None:
    Xraw = np.asarray(X_test, dtype=np.float32)                 # (N, Npoints)
elif (
    'X_test_in' in globals()
    and X_test_in is not None
    and np.asarray(X_test_in).ndim == 3
    and np.asarray(X_test_in).shape[2] == 3
):
    X_test_norm = np.asarray(X_test_in, dtype=np.float32)
    Xraw = None
else:
    raise RuntimeError("[ERROR] Neither raw X_test nor valid 3-channel X_test_in is available. Check Step 2.")

if Xraw is not None:
    N = Xraw.shape[0]
    Npoints = Xraw.shape[1]
    assert len(B_axis) == Npoints, \
        f"[ERROR] B_axis length mismatch: expected {Npoints}, got {len(B_axis)}"

    ch0 = standardize_per_sample(Xraw)                          # (N, Npoints)
    ch1 = ptp_norm_per_sample(Xraw)                             # (N, Npoints)
    ch2 = baxis_channel(B_axis, N)                              # (N, Npoints)
    X_test_norm = np.stack([ch0, ch1, ch2], axis=-1).astype(np.float32)

assert X_test_norm.ndim == 3 and X_test_norm.shape[2] == 3, \
    f"[ERROR] Expected X_test_norm shape (N, Npoints, 3), got {X_test_norm.shape}"

# ---------------------------
# 2) Load latest trained model and normalization limits
# ---------------------------
runs_dir = os.path.join(work_dir, "runs")

def pick_latest_model():
    cands = glob.glob(os.path.join(runs_dir, "*", "*.keras"))
    if not cands:
        cands = glob.glob(os.path.join(work_dir, "*.keras"))
    if not cands:
        raise FileNotFoundError("[ERROR] No .keras model found in runs/ or work_dir.")
    latest = max(cands, key=os.path.getmtime)
    return latest, os.path.dirname(latest)

model_path, run_dir = pick_latest_model()
print(f"[INFO] Loading model from: {model_path}")
model = keras.models.load_model(model_path, compile=False)

out_names = list(model.output_names)
print("[INFO] Model outputs:", out_names)

expected = ['B0', 'dB', 'p3']
if out_names != expected:
    print("[WARN] Model output names/order differ from expected:", out_names)
    print("[WARN] Metrics below assume expected order:", expected)

# Load y_min / y_max
if run_dir and os.path.exists(os.path.join(run_dir, "y_min.npy")):
    y_min_local = np.load(os.path.join(run_dir, "y_min.npy")).astype(np.float32).reshape(1, -1)
    y_max_local = np.load(os.path.join(run_dir, "y_max.npy")).astype(np.float32).reshape(1, -1)
else:
    y_min_local = np.load(os.path.join(work_dir, "y_min.npy")).astype(np.float32).reshape(1, -1)
    y_max_local = np.load(os.path.join(work_dir, "y_max.npy")).astype(np.float32).reshape(1, -1)

assert y_min_local.shape == (1,3) and y_max_local.shape == (1,3), \
    f"[ERROR] Expected y_min/y_max shape (1,3), got {y_min_local.shape}/{y_max_local.shape}"

print("[INFO] Using normalization limits:")
print("       y_min:", y_min_local.flatten())
print("       y_max:", y_max_local.flatten())

# ---------------------------
# 3) Prediction (normalized space)
# ---------------------------
X_test_in = np.asarray(X_test_norm, dtype=np.float32)
pred = model.predict(X_test_in, batch_size=64, verbose=0)

if isinstance(pred, list):
    pred_dict = {name: arr for name, arr in zip(out_names, pred)}
    y_pred_norm = np.column_stack([pred_dict[k] for k in expected]).astype(np.float32)
elif isinstance(pred, dict):
    y_pred_norm = np.column_stack([pred[k] for k in expected]).astype(np.float32)
else:
    y_pred_norm = np.asarray(pred, dtype=np.float32)

y_pred_norm = np.clip(y_pred_norm, 0.0, 1.0)

# ---------------------------
# 4) Denormalization to physical units
# ---------------------------
y_true = y_test_norm.astype(np.float32) * (y_max_local - y_min_local) + y_min_local
y_pred = y_pred_norm                    * (y_max_local - y_min_local) + y_min_local

# ---------------------------
# 5) Metrics (MAE / RMSE in physical units)
# ---------------------------
print("\n[INFO] Test metrics (denormalized, physical units):")
for i, name in enumerate(expected):
    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
    print(f"[INFO] {name:>2}: MAE={mae:.6g}, RMSE={rmse:.6g}")

# ---------------------------
# 6) Parity plots
# ---------------------------
out_dir = run_dir or work_dir
os.makedirs(out_dir, exist_ok=True)

fig = plt.figure(figsize=(15, 4))
for i, name in enumerate(expected):
    ax = fig.add_subplot(1, 3, i+1)
    ax.scatter(y_true[:, i], y_pred[:, i], s=10, alpha=0.35)
    lo = float(min(y_true[:, i].min(), y_pred[:, i].min()))
    hi = float(max(y_true[:, i].max(), y_pred[:, i].max()))
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1)
    ax.set_xlabel(f"True {name}")
    ax.set_ylabel(f"Predicted {name}")
    ax.set_title(f"{name}: True vs Predicted")
    ax.grid(True)

fig.tight_layout()
fig.savefig(os.path.join(out_dir, "parity_test.png"), dpi=150, bbox_inches="tight")
plt.show()

# ---------------------------
# 7) Save predictions to CSV
# ---------------------------
out_csv = os.path.join(out_dir, "dysonian_test_predictions.csv")
np.savetxt(
    out_csv,
    np.column_stack([y_true, y_pred]),
    delimiter=",",
    header="B0_true,dB_true,p3_true,B0_pred,dB_pred,p3_pred",
    comments="",
    fmt="%.8f"
)
print(f"\n[INFO] Predictions saved to: {out_csv}")
print(f"[INFO] Parity plot saved to: {os.path.join(out_dir, 'parity_test.png')}")

# 6. Getting Parameters from Real Spectra



## 6.1. Parameter Inference Setup for Real Spectra

This cell prepares the environment for **extracting Dysonian parameters from real EPR spectra**.  
It safely mounts Google Drive, selects the latest trained model run, and loads the CNN together with the corresponding normalization parameters and magnetic field axis.

Utility functions are defined to construct the correct multi-channel input representation for real spectra, ensuring consistency with the model configuration before prediction and denormalization.

In [ ]:
# Parameter inference from real spectra: safe Drive mount + load latest model + load scalers/axis ===

import os, glob
import numpy as np
import tensorflow as tf
from pathlib import Path
from google.colab import drive

# ---------------------------
# 1) Safe Google Drive mount
# ---------------------------
def safe_mount_gdrive(mountpoint="/content/drive", force=False):
    """Safely mount Google Drive in Colab and return the mount point used."""
    mydrive_path = os.path.join(mountpoint, "MyDrive")

    if os.path.isdir(mydrive_path) and len(os.listdir(mydrive_path)) > 0:
        print(f"[INFO] Google Drive is already mounted at: {mountpoint}")
        return mountpoint

    if os.path.exists(mountpoint) and os.listdir(mountpoint):
        if force:
            print("[WARN] Mount point is not empty; forcing remount.")
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            os.system(f"rm -rf {mountpoint}")
        else:
            print("[INFO] Mount point is not empty; using alternative mount point: /content/gdrive")
            mountpoint = "/content/gdrive"
            os.makedirs(mountpoint, exist_ok=True)

    drive.mount(mountpoint, force_remount=force)

    mydrive_path = os.path.join(mountpoint, "MyDrive")
    if os.path.isdir(mydrive_path):
        print(f"[INFO] Google Drive mounted successfully at: {mountpoint}")
    else:
        raise RuntimeError("[ERROR] Failed to detect the MyDrive folder; check permissions.")
    return mountpoint


# ---------------------------
# 2) Project paths and run selection
# ---------------------------
mp = safe_mount_gdrive("/content/drive", force=False)

# Select project folder here
PROJECT_DIR_NAME = "DysonianLineCNN-mix"   # or "DysonianLineCNN-2"
base_dir = Path(mp) / "MyDrive" / "Python" / PROJECT_DIR_NAME
runs_dir = base_dir / "runs"

assert runs_dir.exists(), f"[ERROR] runs/ not found: {runs_dir}"

def pick_latest_run(runs_dir: Path) -> Path:
    """Select the latest run directory by modification time."""
    run_candidates = [p for p in runs_dir.iterdir() if p.is_dir()]
    if not run_candidates:
        raise FileNotFoundError(f"[ERROR] No run folders found in: {runs_dir}")
    latest = max(run_candidates, key=lambda p: p.stat().st_mtime)
    return latest

# Optional manual override:
# model_dir = runs_dir / "20251216_121657"
# and comment out the next line
model_dir = pick_latest_run(runs_dir)

print(f"[INFO] Using run_dir: {model_dir}")

# ---------------------------
# 3) Load model (.keras) from the selected run_dir
# ---------------------------
keras_files = sorted(model_dir.glob("*.keras"))
if not keras_files:
    raise FileNotFoundError(f"[ERROR] No .keras model found in: {model_dir}")
model_file = keras_files[0]  # if there is a single file; otherwise pick by timestamp
# Prefer newest file by modification time:
model_file = max(keras_files, key=lambda p: p.stat().st_mtime)

model = tf.keras.models.load_model(model_file, compile=False)
print(f"[INFO] CNN model loaded from: {model_file}")

print("[INFO] Model outputs:", list(model.output_names))
print("[INFO] Model input shape:", model.input_shape)  # (None, 4096, C)
in_channels = model.input_shape[-1]
print(f"[INFO] Expected number of channels (C): {in_channels}")

# ---------------------------
# 4) Load y_min/y_max and B_axis from the same run_dir
# ---------------------------
ymin_path = model_dir / "y_min.npy"
ymax_path = model_dir / "y_max.npy"
baxis_path = model_dir / "B_axis.npy"

assert ymin_path.exists() and ymax_path.exists(), "[ERROR] Missing y_min.npy / y_max.npy in run_dir."
y_min = np.load(ymin_path).astype(np.float32).reshape(1, -1)
y_max = np.load(ymax_path).astype(np.float32).reshape(1, -1)

print("[INFO] Loaded y_min/y_max from run_dir.")
print("       y_min:", y_min.flatten())
print("       y_max:", y_max.flatten())

B_axis = None
if baxis_path.exists():
    B_axis = np.load(baxis_path).astype(np.float32)
    print(f"[INFO] Loaded B_axis from run_dir (shape={B_axis.shape}).")
else:
    print("[WARN] B_axis.npy not found in run_dir (expected only if using the axis channel).")

# ---------------------------
# 5) Helper: build model input channels (2-ch or 3-ch)
# ---------------------------
def standardize_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    mu = Xarr.mean(axis=1, keepdims=True)
    sd = Xarr.std(axis=1, keepdims=True) + eps
    return (Xarr - mu) / sd

def ptp_norm_per_sample(Xarr, eps=1e-8):
    Xarr = Xarr.astype(np.float32, copy=False)
    xmin = Xarr.min(axis=1, keepdims=True)
    ptp  = (Xarr.max(axis=1, keepdims=True) - xmin) + eps
    return (Xarr - xmin) / ptp * 2.0 - 1.0  # [-1, 1]

def build_input_channels(Xraw_2d, B_axis=None, expected_C=2):
    """
    Xraw_2d: (N, 4096) real spectra (signal)
    Returns: (N, 4096, C)
    C=2: [standardized, ptp_norm]
    C=3: + [B_axis_norm] (same axis repeated per sample)
    """
    ch0 = standardize_per_sample(Xraw_2d)
    ch1 = ptp_norm_per_sample(Xraw_2d)
    X = np.stack([ch0, ch1], axis=-1)  # (N, 4096, 2)

    if expected_C == 3:
        if B_axis is None:
            raise ValueError("[ERROR] Model expects 3 channels but B_axis is None. Load B_axis.npy or provide B_axis.")
        # Normalize B-axis to [-1, 1] once, then tile to N samples
        b = B_axis.astype(np.float32).reshape(1, -1)
        b = (b - b.min()) / (b.max() - b.min() + 1e-8) * 2.0 - 1.0
        b = np.repeat(b, Xraw_2d.shape[0], axis=0)  # (N, 4096)
        X = np.concatenate([X, b[..., None]], axis=-1)  # (N, 4096, 3)

    if X.shape[-1] != expected_C:
        raise RuntimeError(f"[ERROR] Channel mismatch: got {X.shape[-1]}, expected {expected_C}")
    return X.astype(np.float32)

print("[INFO] Runtime loader initialized. Next: load real spectra -> build_input_channels -> predict -> denormalize.")

## 6.2. Parameter Extraction from a Real EPR Spectrum

This cell applies the trained Dysonian CNN to a **single real EPR spectrum**.  
The spectrum is converted into the same three-channel representation used during training, including the physical magnetic field axis.

The model predicts normalized parameters, which are then denormalized to physical units.  
The predicted values and diagnostic artifacts are saved for further analysis and verification.

In [ ]:
# === Real spectrum -> 3-channel CNN input -> Predict -> Denormalize ===

import os, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from google.colab import drive

# -----------------------
# 0) Safe Google Drive mount
# -----------------------
def safe_mount_gdrive(mountpoint="/content/drive", force=False):
    mydrive_path = os.path.join(mountpoint, "MyDrive")
    if os.path.isdir(mydrive_path) and len(os.listdir(mydrive_path)) > 0:
        print(f"[INFO] Google Drive is already mounted at: {mountpoint}")
        return mountpoint
    if os.path.exists(mountpoint) and os.listdir(mountpoint):
        if force:
            print("[WARN] Mount point is not empty; forcing remount.")
            try:
                drive.flush_and_unmount()
            except Exception:
                pass
            os.system(f"rm -rf {mountpoint}")
        else:
            print("[INFO] Mount point is not empty; using alternative mount point: /content/gdrive")
            mountpoint = "/content/gdrive"
            os.makedirs(mountpoint, exist_ok=True)
    drive.mount(mountpoint, force_remount=force)
    mydrive_path = os.path.join(mountpoint, "MyDrive")
    if os.path.isdir(mydrive_path):
        print(f"[INFO] Google Drive mounted successfully at: {mountpoint}")
    else:
        print("[ERROR] Failed to detect the MyDrive folder; check permissions.")
    return mountpoint

mp = safe_mount_gdrive("/content/drive")

# -----------------------
# 1) Paths (edit as needed)
# -----------------------
base_dir = Path(mp) / "MyDrive" / "Python" / "DysonianLineCNN-mix"
                                        # <<<<<<<<<<<<<<<<<<<<<<<<<<
runName  = "20260125_161258"            # <<< SET RUN NAME HERE
                                        # <<<<<<<<<<<<<<<<<<<<<<<<<<
run_dir  = base_dir / "runs" / runName
assert run_dir.exists(), f"[ERROR] Run directory not found: {run_dir}"

# Model file (single *.keras expected in run_dir)
model_file = next(run_dir.glob("*.keras"))
print("[INFO] Using model:", model_file)

# Normalization files
ymin_file  = run_dir / "y_min.npy"
ymax_file  = run_dir / "y_max.npy"
baxis_file = run_dir / "B_axis.csv"

assert ymin_file.exists() and ymax_file.exists(), "[ERROR] y_min.npy / y_max.npy not found in run_dir."
assert baxis_file.exists(), "[ERROR] B_axis.csv not found in run_dir."

# Real spectrum (exported by MATLAB to project root)
                                        # <<<<<<<<<<<<<<<<<<<<<<<<<<
spec_file = base_dir / "022704_spectrum.csv"  # <<< SET SPECTRUM NAME HERE
                                        # <<<<<<<<<<<<<<<<<<<<<<<<<<
assert spec_file.exists(), f"[ERROR] spectrum.csv not found: {spec_file}"

# Output files
out_json = base_dir / "real_predicted_params.json"
out_png  = base_dir / "real_input_channels_preview.png"

# -----------------------
# 2) Load model, bounds, and B_axis
# -----------------------
model = tf.keras.models.load_model(model_file, compile=False)
print("[INFO] Model input shape:", model.input_shape)
print("[INFO] Model outputs:", model.output_names)

y_min = np.load(ymin_file).astype(np.float32).reshape(1, -1)
y_max = np.load(ymax_file).astype(np.float32).reshape(1, -1)
assert y_min.shape == (1,3) and y_max.shape == (1,3), \
    f"[ERROR] y_min/y_max must be (1,3), got {y_min.shape}/{y_max.shape}"

B_axis = np.loadtxt(baxis_file, delimiter=",").astype(np.float32).reshape(-1)
assert B_axis.shape[0] == 4096, f"[ERROR] B_axis must contain 4096 points, got {B_axis.shape}"

# -----------------------
# 3) Load real spectrum (4096 points)
# -----------------------
x = np.loadtxt(spec_file, delimiter=",").astype(np.float32).reshape(-1)
assert x.shape[0] == 4096, f"[ERROR] Spectrum must contain 4096 points, got {x.shape}"

# -----------------------
# 4) Build 3 input channels
#    ch0: per-sample standardization
#    ch1: per-sample ptp normalization to [-1,1]
#    ch2: positional channel from B_axis normalized to [-1,1]
# -----------------------
eps = 1e-8

# Channel 0: z-score
mu = x.mean()
sd = x.std() + eps
ch0 = (x - mu) / sd

# Channel 1: ptp normalization
xmin = x.min()
xptp = (x.max() - xmin) + eps
ch1 = (x - xmin) / xptp * 2.0 - 1.0

# Channel 2: B-axis positional encoding
bmin = B_axis.min()
bptp = (B_axis.max() - bmin) + eps
ch2 = (B_axis - bmin) / bptp * 2.0 - 1.0

X_in = np.stack([ch0, ch1, ch2], axis=-1).astype(np.float32)  # (4096,3)
X_in = X_in[None, ...]                                        # (1,4096,3)

print("[INFO] Prepared model input X_in with shape:", X_in.shape)

# -----------------------
# 5) Predict (normalized) and denormalize
# -----------------------
pred = model.predict(X_in, verbose=0)

out_names = list(model.output_names)

if isinstance(pred, list):
    y_pred_norm = np.concatenate([p.reshape(1,1) for p in pred], axis=1).astype(np.float32)
elif isinstance(pred, dict):
    y_pred_norm = np.column_stack([pred[k].reshape(-1) for k in out_names]).astype(np.float32)
else:
    y_pred_norm = np.asarray(pred, dtype=np.float32).reshape(1, -1)

y_pred_norm = np.clip(y_pred_norm, 0.0, 1.0)
y_pred = y_pred_norm * (y_max - y_min) + y_min

# -----------------------
# 6) Report results and save artifacts
# -----------------------
result = {
    "runName": runName,
    "model_file": str(model_file),
    "output_names": out_names,
    "y_pred_norm": y_pred_norm.flatten().tolist(),
    "y_pred_physical": {out_names[i]: float(y_pred[0, i]) for i in range(3)},
    "notes": "Input channels: [standardize, ptp(-1..1), position(B_axis->-1..1)]."
}

print("\n[INFO] Predicted parameters (physical units):")
for k, v in result["y_pred_physical"].items():
    print(f"[INFO] {k}: {v:.6g}")

with open(out_json, "w") as f:
    json.dump(result, f, indent=2)
print("[INFO] Saved results to:", out_json)

# Preview input channels
plt.figure(figsize=(12,3.5))
plt.plot(ch0, label="ch0 (z-score)")
plt.plot(ch1, label="ch1 (ptp [-1,1])")
plt.plot(ch2, label="ch2 (B-axis position [-1,1])")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print("[INFO] Saved input channel preview to:", out_png)

# 7. Google Drive Unmount and Cleanup

This cell safely **unmounts Google Drive** from the Colab runtime if it is currently mounted.  
It flushes cached data before unmounting to ensure a clean detachment and reports the operation status, avoiding redundant actions when Drive is not mounted.

In [ ]:
import os
from google.colab import drive

drive_path = "/content/drive"

# --- Check mount status ---
if os.path.ismount(drive_path):
    print("[INFO] Google Drive is currently mounted. Attempting to unmount.")

    try:
        drive.flush_and_unmount()  # Safe unmount with cache flush
        print("[INFO] Google Drive has been successfully unmounted.")
    except Exception as e:
        print(f"[ERROR] Failed to unmount Google Drive: {e}")
else:
    print("[INFO] Google Drive is not mounted. No action required.")